# Generic Training Notebook — EVM-nano (VOC → COCO → Export → Bench)

Platform-agnostic twin of `Colab_Notebook.ipynb` — **same stages**:

**overfit gate → VOC full → full COCO → ONNX/INT8 export → latency bench → curves**

Primary target: **Kaggle**, driven headlessly via the Kaggle CLI
(`kaggle kernels push` → `kaggle kernels status` → `kaggle kernels output`),
so results flow back to the dev machine without manual copy-paste.

- Kaggle: choose **GPU T4 x1** (single-GPU by design), **Internet: On**
- Artifacts land in `/kaggle/working` (= kernel output) — that is what
  `kaggle kernels output` delivers back
- Full COCO spans Kaggle's 12h session cap: each new kernel version re-attaches the
  previous output and the **resume relay** (§4) picks up `last.pt` automatically
- Overfit gate (mAP@0.5 ≥ 0.90) **must pass** before any full training


In [ ]:
import os
import sys
KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB = 'google.colab' in sys.modules
if KAGGLE:
    WORK = '/kaggle/working'
    DATA = '/kaggle/tmp/datasets'
    REPO = '/kaggle/tmp/edge-vision-model'
    RUNS = '/kaggle/working/runs'
elif IN_COLAB:
    WORK = '/content'
    DATA = '/content/datasets'
    REPO = '/content/edge-vision-model'
    RUNS = '/content/runs'
else:
    WORK = '.'
    DATA = 'datasets'
    REPO = '.'
    RUNS = 'runs'
os.makedirs(DATA, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
print('kaggle:', KAGGLE, '| colab:', IN_COLAB)
print('repo:', REPO, '| data:', DATA, '| runs:', RUNS)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)


## 1. Setup — repo + deps

Kaggle: Settings → Internet → **On** (needed for clone, pip, dataset mirrors).


In [ ]:
import subprocess
import sys
if not os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', 'clone',
                   'https://github.com/avneeshjadhav04/edge-vision-model', REPO],
                  check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
sys.path.insert(0, REPO)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'onnx',
                'onnxruntime', 'onnxsim', 'opencv-python-headless'])
print('repo ready at', REPO)


## 2. VOC data (official mirrors, runtime download)


In [ ]:
from data.download import download_voc
import glob
download_voc(root=f'{DATA}/VOC')
for f in glob.glob(f'{DATA}/VOC/*.tar'):
    os.remove(f)
print('VOC ready at', f'{DATA}/VOC')


## 3. Overfit sanity gate — 20 images, mAP@0.5 ≥ 0.90 or STOP

Fast correctness check (~5–10 min on T4). If this fails, do **not** start full training.


In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'scripts.overfit_test', '--root', f'{DATA}/VOC',
       '--epochs', '300', '--batch-size', '8', '--img-size', '320',
       '--device', DEVICE, '--save-dir', f'{RUNS}/overfit']
print(' '.join(cmd))
rc = subprocess.run(cmd).returncode
os.makedirs(f'{RUNS}/overfit', exist_ok=True)
gate = 'PASS' if rc == 0 else 'FAIL'
with open(f'{RUNS}/overfit/GATE', 'w') as f:
    f.write(gate)
print('GATE:', gate)
assert gate == 'PASS', 'Overfit gate FAILED — fix the model/pipeline before full training'


## 4. VOC full training (120 epochs) — resume-aware

Kaggle resume relay first: a previous kernel version's output is attached at
`/kaggle/input/<slug>/`; its `last.pt`/`best.pt`/`log.json` are copied forward.


In [ ]:
if KAGGLE:
    import glob
    import shutil
    for run in ('voc', 'coco'):
        dst = os.path.join(RUNS, run)
        os.makedirs(dst, exist_ok=True)
        for name in ('last.pt', 'best.pt', 'log.json'):
            hits = sorted(glob.glob(f'/kaggle/input/*/runs/{run}/{name}'))
            if hits and not os.path.exists(os.path.join(dst, name)):
                shutil.copy(hits[-1], os.path.join(dst, name))
                print('relayed', run, name, 'from', hits[-1])
    print('resume relay done')


In [ ]:
resume = f'--resume {RUNS}/voc/last.pt' if os.path.exists(f'{RUNS}/voc/last.pt') else ''
cmd = (f'python -m scripts.train --dataset voc --root {DATA}/VOC '
       f'--epochs 120 --batch-size 32 --img-size 640 --device {DEVICE} '
       f'--save-dir {RUNS}/voc {resume}')
print(cmd)
rc = subprocess.run(cmd, shell=True).returncode
print('rc =', rc, '(nonzero after a session timeout is expected — push a new version to resume)')


## 5. COCO — data + full 300-epoch training (runs by default)

Full schedule, resume across 12h Kaggle sessions via the relay above.
Each new kernel version re-downloads COCO (~20–30 min) then continues from `last.pt`.


In [ ]:
from data.download import download_coco
download_coco(root=f'{DATA}/coco', splits=('val2017',), with_train=True)
for f in glob.glob(f'{DATA}/coco/*.zip'):
    os.remove(f)
print('COCO ready at', f'{DATA}/coco')


In [ ]:
init = f'--init-from {RUNS}/voc/best.pt' if os.path.exists(f'{RUNS}/voc/best.pt') else ''
resume = f'--resume {RUNS}/coco/last.pt' if os.path.exists(f'{RUNS}/coco/last.pt') else ''
cmd = (f'python -m scripts.train --dataset coco --root {DATA}/coco '
       f'--epochs 300 --batch-size 64 --img-size 640 --device {DEVICE} '
       f'--save-dir {RUNS}/coco {init} {resume}')
print(cmd)
rc = subprocess.run(cmd, shell=True).returncode
print('rc =', rc, '(nonzero after a session timeout is expected — push a new version to resume)')


## 6. Evals — VOC2007 test + COCO val2017


In [ ]:
for ds, root in (('voc', f'{DATA}/VOC'), ('coco', f'{DATA}/coco')):
    w = os.path.join(RUNS, ds, 'best.pt')
    if not os.path.exists(w):
        print(ds, 'eval skipped (no best.pt)')
        continue
    cmd = (f'python -m scripts.eval --dataset {ds} --root {root} '
           f'--weights {w} --img-size 640 --device {DEVICE}')
    print(cmd)
    subprocess.run(cmd, shell=True)


## 7. Export — ONNX (aux stripped) + INT8 + torch↔ORT parity


In [ ]:
BEST = f'{RUNS}/coco/best.pt' if os.path.exists(f'{RUNS}/coco/best.pt') else f'{RUNS}/voc/best.pt'
NC = 80 if 'coco' in BEST else 20
IMG = 640
print('exporting from', BEST, '| classes:', NC)
os.makedirs(f'{RUNS}/export', exist_ok=True)
cmd = (f'python -m export.onnx_export --weights {BEST} --out {RUNS}/export/evm_nano.onnx '
       f'--num-classes {NC} --img-size {IMG}')
print(cmd)
assert subprocess.run(cmd, shell=True).returncode == 0
from export.quantize import quantize_int8
quantize_int8(f'{RUNS}/export/evm_nano.onnx', f'{RUNS}/export/evm_nano_int8.onnx', img_size=IMG)
print('export done')


In [ ]:
import numpy as np
import torch
import onnxruntime as ort
from models import build_model
from export.decode_onnx import decode_outputs
from scripts.common import load_config
m = build_model(load_config('model_nano'), num_classes=NC)
sd = torch.load(BEST, map_location='cpu', weights_only=False)
m.load_state_dict(sd.get('model', sd), strict=True)
m.eval()
x = torch.randn(1, 3, IMG, IMG)
with torch.no_grad():
    res_t = m.predict(x, score_thresh=0.0, max_det=1000, use_obj=True)
sess = ort.InferenceSession(f'{RUNS}/export/evm_nano.onnx', providers=['CPUExecutionProvider'])
raw = sess.run(None, {'images': x.numpy()})
res_o = decode_outputs(raw, IMG, num_classes=NC, score_thresh=0.0, max_det=1000)
st = res_t[0]['scores'].numpy()
so = res_o[0]['scores']
print('max score torch / onnx:', float(st.max()), float(so.max()))
assert abs(float(st.max()) - float(so.max())) < 5e-3
print('PARITY OK')


## 8. Latency benchmarks

Hosted (Kaggle/Colab) CPU numbers are **indicative**. A local run of this notebook on the
target laptop produces the real latency-vs-mAP table.


In [ ]:
cmd = (f'python -m benchmarks.bench_runtime --onnx {RUNS}/export/evm_nano.onnx '
       f'--img-size {IMG} --n-iter 50 --int8')
print(cmd)
subprocess.run(cmd, shell=True)


## 9. Curves + metrics.json (→ kernel output)

`metrics.json`, `log.json`, curves, checkpoints and the exported models all live under
`RUNS` (= kernel output on Kaggle) — `kaggle kernels output` delivers them to the dev
machine, closing the loop without manual copy-paste.


In [ ]:
import json
import matplotlib.pyplot as plt
def curve(log_json, title):
    h = json.load(open(log_json))
    ep = [x['epoch'] for x in h]
    loss = [x['loss'] for x in h]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(ep, loss)
    ax[0].set_title(f'{title} loss')
    ax[0].set_xlabel('epoch')
    m = [(x['epoch'], x['mAP']) for x in h if 'mAP' in x]
    if m:
        ax[1].plot(*zip(*m))
        ax[1].set_title(f'{title} mAP')
        ax[1].set_xlabel('epoch')
    plt.tight_layout()
    plt.savefig(f'{RUNS}/{title.lower()}_curve.png', dpi=150)
    plt.show()
for ds in ('voc', 'coco'):
    try:
        curve(f'{RUNS}/{ds}/log.json', ds.upper())
    except FileNotFoundError:
        print(ds, 'log missing')


In [ ]:
metrics = {}
for ds, total_ep in (('voc', 120), ('coco', 300)):
    try:
        h = json.load(open(f'{RUNS}/{ds}/log.json'))
        maps = [x['mAP'] for x in h if 'mAP' in x]
        metrics[ds] = {'best_mAP': max(maps) if maps else None,
                      'epochs_done': h[-1]['epoch'] + 1,
                      'complete': h[-1]['epoch'] + 1 >= total_ep}
    except FileNotFoundError:
        metrics[ds] = None
for p in glob.glob(f'{RUNS}/export/*.onnx'):
    metrics[os.path.basename(p)] = os.path.getsize(p)
with open(f'{RUNS}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=1)
print(json.dumps(metrics, indent=1))
